# Course notebook — Knowledge and Retrieval Systems

Build grounded knowledge systems whose ingestion, retrieval, generation, and citations can be evaluated separately.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Knowledge strategy and ingestion


### Lecture 01 · Knowledge Outside the Model

Decide among direct context, search, databases, knowledge graphs, RAG, fine-tuning, and deterministic rules.

**Predict before running.** Put knowledge in the component best suited to update, govern, query, and verify it. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Classify ten requirements by the correct knowledge mechanism.


In [ ]:
"""Lab 6.1: Knowledge Outside the Model"""

CHAPTER = "6.1"
print("chapter hook:", CHAPTER)
requirements = [
    ("cite exact page", "RAG"),
    ("friendly tone", "prompt/finetune"),
    ("live headcount", "SQL tool"),
]
for req, mechanism in requirements:
    print({"requirement": req, "mechanism": mechanism})
print("---")
print("change one input above, predict output, re-run")


### Lecture 02 · Document Ingestion

Preserve provenance while parsing documents, OCR, tables, images, metadata, permissions, versions, and deletions.

**Predict before running.** Retrieval cannot recover content or permissions lost during ingestion. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create an ingestion manifest and measure parse fidelity.


In [ ]:
"""Lab 6.2: Document Ingestion"""

manifest = {
    "doc_id": "POL-441",
    "pages": 12,
    "acl": "HR-ONLY",
    "chunks": [{"id": 1, "page": 3, "text_hash": "abc123"}],
}
def allowed(chunk, user_groups):
    required = manifest["acl"]
    return required in user_groups
user = {"groups": ["ALL-STAFF"]}
print({"access": allowed(manifest["chunks"][0], user["groups"])})


**Section checkpoint.** Sketch source identifiers and chunk lineage for two versions of the same policy.


## Retrieval and reranking


### Lecture 03 · Retrieval

Compare lexical, dense, sparse, hybrid, filtered, multi-query, parent-child, and late-interaction retrieval.

**Predict before running.** Retrieval is candidate selection under relevance and policy constraints. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement lexical and vector baselines and calculate recall@k.


In [ ]:
"""Lab 6.3: Retrieval"""

CHAPTER = "6.3"
print("chapter hook:", CHAPTER)
docs = {"a": "PTO accrual cap is 240 hours", "b": "Leave policy overview"}
query = set("pto cap".split())
scores = {k: len(query & set(v.lower().split())) for k, v in docs.items()}
print("bm25_proxy:", sorted(scores.items(), key=lambda x: -x[1]))
print("---")
print("change one input above, predict output, re-run")


### Lecture 04 · Ranking and Context Selection

Use fusion, reranking, diversity, deduplication, compression, and token-aware packing.

**Predict before running.** Every selected passage competes for limited attention; more context can reduce quality. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Add reranking and measure quality versus latency.


In [ ]:
"""Lab 6.4: Ranking and Context Selection"""

rank_a = ["doc-leave", "doc-expense", "doc-security"]
rank_b = ["doc-expense", "doc-leave", "doc-onboarding"]
def rrf(lists, k=60):
    scores = {}
    for ranking in lists:
        for rank, doc in enumerate(ranking, 1):
            scores[doc] = scores.get(doc, 0) + 1/(k+rank)
    return sorted(scores.items(), key=lambda x: -x[1])
print("rrf top2:", rrf([rank_a, rank_b])[:2])


**Section checkpoint.** Compare two ranked lists and trace how reciprocal rank fusion combines them.


## Grounded answers and enterprise constraints


### Lecture 05 · RAG Generation and Citations

Construct grounded prompts, handle missing evidence, attribute claims, validate citations, and avoid unsupported synthesis.

**Predict before running.** A citation is useful only when it supports the nearby claim and resolves to source evidence. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Build a citation validator that checks claim-to-source alignment.


In [ ]:
"""Lab 6.5: RAG Generation and Citations"""

CHAPTER = "6.5"
print("chapter hook:", CHAPTER)
claim = "PTO cap is 300 hours"
source = "PTO accrual cap is 240 hours"
claim_tokens = set(claim.lower().split())
source_tokens = set(source.lower().split())
overlap = len(claim_tokens & source_tokens) / len(claim_tokens)
print({"overlap": round(overlap, 2), "supported": "240" in source and "240" in claim})
print("---")
print("change one input above, predict output, re-run")


### Lecture 06 · Advanced and Enterprise RAG

Study graph, multi-hop, adaptive, and agentic retrieval together with tenancy, freshness, security, resilience, and cost.

**Predict before running.** Advanced orchestration cannot compensate for weak data, retrieval, authorization, or evaluation. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Complete the enterprise RAG architecture studio and threat model.


In [ ]:
"""Lab 6.6: Advanced and Enterprise RAG"""

CHAPTER = "6.6"
print("chapter hook:", CHAPTER)
routes = {"simple": {"hops": 1, "cost": 1}, "multi": {"hops": 3, "cost": 4}}
def adaptive(query):
    return "multi" if "and also" in query or "depending on" in query else "simple"
queries = ["PTO cap?", "PTO cap and carryover depending on tenure"]
for q in queries:
    r = adaptive(q)
    print({"query": q, "route": r, "cost_units": routes[r]["cost"]})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Define pass cases for authorized evidence, no relevant evidence, and a citation to the wrong passage.


## Course workshop

Build rankings with one shared candidate. Test how agreement affects rank before evaluating citation faithfulness separately.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Fuse lexical and vector-like rankings with reciprocal rank fusion."""
LEXICAL = ["doc-expense", "doc-leave", "doc-security"]
VECTOR = ["doc-leave", "doc-expense", "doc-onboarding"]


def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, document in enumerate(ranking, start=1):
            scores[document] = scores.get(document, 0.0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)


for document, score in reciprocal_rank_fusion([LEXICAL, VECTOR]):
    print(f"{score:.5f} {document}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
ranked = reciprocal_rank_fusion([["shared", "lexical"], ["shared", "dense"]])
assert ranked[0][0] == "shared"
print('Workshop checks passed')


## Applied exercise for Lecture 05 — Evaluate grounded answers against an access boundary

Read the labeled cases before you run the cell. Predict which sources an employee should see. Then add a case for a user who has no authorized role. Check source IDs separately from answer text. The generator here returns a passage extract, so you can inspect the retrieval and citation behavior directly.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Passage:
    source: str
    tags: set[str]
    text: str
    roles: set[str]


corpus = [
    Passage("travel", {"travel", "receipt"}, "Attach travel receipts to the expense form.", {"employee", "payroll"}),
    Passage("rates", {"payroll", "rate"}, "Use the current internal payroll rate table.", {"payroll"}),
]


def grounded_answer(query_terms, role):
    allowed = [passage for passage in corpus if role in passage.roles]
    ranked = sorted(allowed, key=lambda passage: len(query_terms & passage.tags), reverse=True)
    if not ranked or not query_terms & ranked[0].tags:
        return "No authorized matching evidence.", []
    evidence = ranked[0]
    return evidence.text, [evidence.source]


cases = [
    ({"travel", "receipt"}, "employee", ["travel"]),
    ({"payroll", "rate"}, "employee", []),
    ({"weather"}, "employee", []),
    ({"payroll", "rate"}, "payroll", ["rates"]),
]
for terms, role, expected_sources in cases:
    answer, sources = grounded_answer(terms, role)
    assert sources == expected_sources, (terms, role, sources)
    if not expected_sources:
        assert answer == "No authorized matching evidence."
    print(role, sorted(terms), answer, sources)


## Course project

Deliver an enterprise RAG system with authorization, hybrid retrieval, reranking, citations, and stage-specific evaluation.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
